# SciGraphAgent Benchmark — Notebook 00
## Setup, Foundations, and Everything We Have Done So Far

---

**Who this notebook is for:**  
A complete beginner. You do not need to know Python, AI, or anything technical before reading this. Every new word, idea, and piece of code is explained from scratch.

**What this notebook covers:**  
Everything we set up in the terminal — explained step by step, with the reasoning behind each decision.

**How to use this notebook:**  
Read each text section first. Then run the code cell below it. Do not skip ahead — each cell builds on the one before it.

---

## Table of Contents

1. What is this project and why does it matter?
2. What is Python and why are we using it?
3. What is a terminal and what did we do in it?
4. What is Git and GitHub?
5. What is a virtual environment?
6. What are libraries and why do we install them?
7. What is an API?
8. What is an LLM?
9. What is Groq and why are we using it?
10. Verifying your complete setup
11. Understanding the project folder structure
12. What comes next?

---
## Section 1 — What is this project and why does it matter?

### The problem we are solving

Imagine you are a scientist. Every day, thousands of new research papers are published in your field. You cannot read all of them. But the answer to your research question might require combining a fact from paper A, a method from paper B, and a finding from paper C — all written by different authors who have never met.

A standard AI assistant would fail at this. If you ask it a question, it looks for text that is *similar* to your question. It finds passages that use similar words. But it cannot reason about how ideas *connect* across many documents.

### Our solution — SciGraphAgent

SciGraphAgent is an AI system that:

1. **Reads scientific papers** and extracts the important entities (methods, datasets, concepts) and the relationships between them
2. **Builds a knowledge graph** — a map of how all these entities connect
3. **Answers questions** by traversing this map, combining graph reasoning with text search
4. **Checks its own answers** — if the answer is not trustworthy, it tries again

### Why this is novel

The key innovation is Step 4 — the system evaluates its own answer using a quality score called **faithfulness**, and if the score is too low, it automatically retrieves more context and tries again. No published scientific AI system does this as a live component inside the agent.

### What this notebook teaches you

Before we can build and test this system, we need to set up the environment — install the tools, connect to the AI service, and organise the project. This notebook explains every step of that setup, from scratch.

---
## Section 2 — What is Python and why are we using it?

### What is a programming language?

A computer only understands electrical signals — ones and zeros. A programming language is a way for humans to write instructions that a computer can eventually understand. Instead of writing ones and zeros, you write words and symbols that follow specific rules.

### What is Python?

Python is a programming language created by Guido van Rossum in 1991. It is designed to be readable — it looks almost like plain English. Compare these two ways of saying "print the word hello":

- In assembly language (close to machine code): `MOV AH, 09h` `MOV DX, OFFSET msg` `INT 21h`
- In Python: `print("hello")`

Python is the dominant language for AI and data science because:
- It is easy to read and write
- It has a massive collection of ready-made tools (called libraries) for AI, data, and science
- Almost every AI company provides Python support first

### Checking your Python version

In [1]:
# In Python, a line starting with # is a comment.
# Comments are notes for humans. Python ignores them completely.
# We use comments throughout this notebook to explain what each line does.

# The 'import' keyword loads a library into your program.
# 'sys' is a built-in library that gives information about the Python system.
import sys

# sys.version is a variable that stores the Python version as text.
# print() displays whatever is inside the brackets.
print("Python version:", sys.version)

# sys.executable is the full path to the Python program being used right now.
print("Python location:", sys.executable)

Python version: 3.14.4 (main, Jun 18 2026, 14:25:02) [GCC 15.2.0]
Python location: /run/media/bala/HDD/Projects/scigraphagent-benchmark/.venv/bin/python


### Basic Python concepts you need to know

Before going further, here are the five most important Python concepts used in this project. Run each cell and read the output.

In [2]:
# ── CONCEPT 1: Variables ──────────────────────────────────────────
# A variable is a named container that holds a value.
# You create one by writing: name = value

project_name = "SciGraphAgent"   # stores text (called a 'string')
num_questions = 50               # stores a whole number (called an 'integer')
alpha = 0.6                      # stores a decimal number (called a 'float')
is_ready = True                  # stores True or False (called a 'boolean')

print("Project:", project_name)
print("Questions:", num_questions)
print("Alpha weight:", alpha)
print("Ready:", is_ready)

Project: SciGraphAgent
Questions: 50
Alpha weight: 0.6
Ready: True


In [3]:
# ── CONCEPT 2: Lists and Dictionaries ────────────────────────────

# A LIST is an ordered collection of items, written with square brackets [].
# This is how we store multiple questions, answers, or results.
datasets = ["hotpotqa", "musique", "2wikimultihopqa"]
print("Datasets:", datasets)
print("First dataset:", datasets[0])   # counting starts at 0, not 1
print("Number of datasets:", len(datasets))

print()

# A DICTIONARY stores key-value pairs, written with curly braces {}.
# Think of it like a real dictionary: look up a word (key) to get its meaning (value).
# This is how we store one benchmark question.
question = {
    "id":       "q001",
    "question": "Which scientist developed the theory of relativity?",
    "answer":   "Albert Einstein",
    "dataset":  "hotpotqa"
}
print("Question:", question["question"])
print("Answer:",   question["answer"])

Datasets: ['hotpotqa', 'musique', '2wikimultihopqa']
First dataset: hotpotqa
Number of datasets: 3

Question: Which scientist developed the theory of relativity?
Answer: Albert Einstein


In [4]:
# ── CONCEPT 3: Functions ──────────────────────────────────────────
# A function is a reusable block of code with a name.
# You DEFINE it once with 'def', then CALL it as many times as you need.

# Defining a function:
def greet(name):
    """This is a docstring — a description of what the function does."""
    message = "Hello, " + name + "!"
    return message   # 'return' sends a value back to whoever called the function

# Calling the function:
result = greet("DrBala")
print(result)

# A more relevant example — a function that counts tokens (roughly)
def estimate_tokens(text):
    """Rough estimate: 1 token ≈ 4 characters in English text."""
    return len(text) // 4   # '//' means integer division (no decimals)

sample_text = "What is a knowledge graph and how does it help with retrieval?"
print(f"Text: '{sample_text}'")
print(f"Estimated tokens: {estimate_tokens(sample_text)}")
# Note: f"..." is an f-string. Anything inside {} is evaluated as Python code.

Hello, DrBala!
Text: 'What is a knowledge graph and how does it help with retrieval?'
Estimated tokens: 15


In [5]:
# ── CONCEPT 4: Loops ──────────────────────────────────────────────
# A loop repeats code. We use loops to process many questions one by one.

# FOR loop — runs once for each item in a list
conditions = ["A: No retrieval", "B: Vector only", "C: Graph only", "D: Hybrid"]

print("Experimental conditions:")
for condition in conditions:
    print(" -", condition)

print()

# Loops with index — enumerate() gives you position AND value
print("Indexed:")
for i, condition in enumerate(conditions):
    print(f"  Condition {i}: {condition}")

Experimental conditions:
 - A: No retrieval
 - B: Vector only
 - C: Graph only
 - D: Hybrid

Indexed:
  Condition 0: A: No retrieval
  Condition 1: B: Vector only
  Condition 2: C: Graph only
  Condition 3: D: Hybrid


In [6]:
# ── CONCEPT 5: Working with files and paths ───────────────────────
# Python's 'pathlib' library makes working with file paths easy.
# A 'Path' object represents a location on your disk.

from pathlib import Path

# Path('.') means 'the current directory'
current_dir = Path('.')
print("Current directory:", current_dir.resolve())  # .resolve() gives the full path

# List files in the current directory
print("\nFiles here:")
for item in sorted(current_dir.iterdir()):
    kind = "[folder]" if item.is_dir() else "[file]  "
    print(f"  {kind} {item.name}")

Current directory: /run/media/bala/HDD/Projects/scigraphagent-benchmark

Files here:
  [file]   .env
  [file]   .env.example
  [folder] .git
  [file]   .gitignore
  [folder] .venv
  [file]   00_setup_and_foundations.ipynb
  [file]   README.md
  [folder] data
  [folder] index
  [file]   requirements.txt
  [folder] results


---
## Section 3 — What is a terminal and what did we do in it?

### What is a terminal?

A terminal (also called a command line or shell) is a text-based way to control your computer. Instead of clicking on icons, you type commands.

On Ubuntu Linux, the terminal runs a program called **bash**. Every command you type is read by bash, executed, and the result is printed back to you.

### Why do developers use the terminal?

Many developer tools — including Git, Python package managers, and AI tools — are designed to be used from the terminal. The terminal also lets you do things much faster than clicking through menus.

### The commands we ran and what they did

Here is every terminal command from our setup session, explained:

```bash
# Navigate into the Projects folder
cd ~/Projects
# 'cd' = change directory. '~' = your home folder. This moves you into Projects.

# Create the project folder
mkdir scigraphagent-benchmark
# 'mkdir' = make directory. Creates a new empty folder.

# Move into it
cd scigraphagent-benchmark

# Print the current location
pwd
# 'pwd' = print working directory. Tells you exactly where you are.
```

### Checking your location from Python

Python can also tell you where it is running from:

In [7]:
import os
from pathlib import Path

# os.getcwd() = get current working directory (same as 'pwd' in terminal)
print("Running from:", os.getcwd())

# Check the project folder exists
project_root = Path(os.getcwd())
print("Project root:", project_root)
print("Folder name:", project_root.name)

# Verify we are inside the right project
if "scigraphagent-benchmark" in str(project_root):
    print("✓ You are inside the correct project folder")
else:
    print("⚠  Warning: you may be in the wrong folder")
    print("   Expected: .../scigraphagent-benchmark")
    print("   Got:     ", project_root)

Running from: /run/media/bala/HDD/Projects/scigraphagent-benchmark
Project root: /run/media/bala/HDD/Projects/scigraphagent-benchmark
Folder name: scigraphagent-benchmark
✓ You are inside the correct project folder


---
## Section 4 — What is Git and GitHub?

### The problem Git solves

Imagine writing a 50-page essay. You save it as `essay.docx`. Then you make changes and save again — overwriting the old version. A week later you realise the changes were wrong. The old version is gone.

Developers face this problem constantly, but with code instead of essays. Git solves it.

### What is Git?

Git is a **version control system**. Every time you tell Git to save a snapshot of your code (called a **commit**), it stores the complete state of every file at that moment. You can go back to any snapshot at any time. Nothing is ever lost.

Git stores all these snapshots in a hidden folder called `.git/` inside your project.

### What is GitHub?

GitHub is a website that stores Git repositories online. Think of it as Google Drive, but specifically designed for code. By pushing your code to GitHub, you:
- Have an online backup
- Can access your code from any computer
- Can share your code with others

### The Git commands we ran

```bash
# Initialise a new Git repository in the current folder
git init
# This creates the hidden .git/ folder. Git starts tracking this folder.

# Tell Git your name and email (stored with every commit you make)
git config --global user.name "DrBala-2-0"
git config --global user.email "312499805+DrBala-2-0@users.noreply.github.com"

# Check what Git knows about the current state
git status

# Stage a file (tell Git: include this file in the next snapshot)
git add README.md

# Create a snapshot with a message describing what changed
git commit -m "Initial commit: add README"

# Send the snapshot to GitHub
git push
```

### The three-stage Git workflow

```
Your files          Staging area         Repository (.git/)      GitHub
(working)     →    (git add)      →      (git commit)      →   (git push)

You edit files   You choose which     Git saves a           GitHub stores
                 files to include     permanent snapshot    it online
```

### Checking Git from Python

In [8]:
import subprocess

# subprocess.run() lets Python run terminal commands and capture the output

def run_command(command):
    """Run a terminal command and return its output as text."""
    result = subprocess.run(
        command,
        shell=True,           # run through the shell (like typing in terminal)
        capture_output=True,  # capture the output instead of printing it
        text=True             # return output as text (not bytes)
    )
    return result.stdout.strip()

# Check Git version
print("Git version:", run_command("git --version"))

# Check current branch
print("Current branch:", run_command("git branch --show-current"))

# Show commit history
print("\nCommit history:")
log = run_command("git log --oneline")
for line in log.split("\n"):
    print(" ", line)

# Show remote (GitHub) connection
print("\nGitHub connection:")
print(" ", run_command("git remote get-url origin"))

Git version: git version 2.53.0
Current branch: main

Commit history:
  8912278 Add .env.example: documents required environment variables
  c3f1131 Add requirements.txt: pinned dependencies with torch CPU install note
  f800d40 Add .gitignore: exclude venv, data, index, results, API keys
  93499cb Initial commit: add README

GitHub connection:
  https://github.com/DrBala-2-0/scigraphagent-benchmark.git


---
## Section 5 — What is a virtual environment?

### The problem it solves

Imagine you have two Python projects on your computer:
- Project A needs version 1.0 of a library called `openai`
- Project B needs version 3.0 of the same library

If you install both on your system, they conflict. Installing version 3.0 overwrites version 1.0, and Project A breaks.

### What is a virtual environment?

A virtual environment is an isolated copy of Python, just for one project. Each project gets its own Python and its own set of libraries that do not interfere with anything else on your computer.

Think of it like a clean room in a laboratory — everything inside is controlled and isolated from the outside world.

### How we created it

```bash
# Create a virtual environment called .venv inside the project folder
python3 -m venv .venv
# '-m venv' means: run the built-in 'venv' module
# '.venv' is the folder name (the dot makes it hidden on Linux)

# Activate it — tell the terminal to use this isolated Python
source .venv/bin/activate
# After this, your prompt shows (.venv) to remind you it is active
```

### Checking the virtual environment

In [9]:
import sys
from pathlib import Path

# sys.prefix is where Python is installed for the current environment
python_location = Path(sys.prefix)
print("Python environment:", python_location)

# Check if we are inside a virtual environment
# A venv always has a 'pyvenv.cfg' file inside it
is_venv = (python_location / "pyvenv.cfg").exists()

if is_venv:
    print("✓ Running inside a virtual environment")
    print("  Venv name:", python_location.name)
    
    # Read the venv config to show what Python it uses
    config_path = python_location / "pyvenv.cfg"
    print("\nVenv configuration:")
    with open(config_path) as f:
        for line in f:
            print(" ", line.strip())
else:
    print("⚠  Not running inside a virtual environment")
    print("   Activate with: source .venv/bin/activate")

Python environment: /run/media/bala/HDD/Projects/scigraphagent-benchmark/.venv
✓ Running inside a virtual environment
  Venv name: .venv

Venv configuration:
  home = /usr/bin
  include-system-site-packages = false
  version = 3.14.4
  executable = /usr/bin/python3.14
  command = /usr/bin/python3 -m venv /run/media/bala/HDD/Projects/scigraphagent-benchmark/.venv


---
## Section 6 — What are libraries and why do we install them?

### What is a library?

A library (also called a package or module) is a collection of pre-written code that solves a specific problem. Instead of writing everything from scratch, you use what someone else already built and tested.

For example:
- Calculating the square root of a number involves complex mathematics. The `math` library has already implemented it. You just call `math.sqrt(16)` and get `4.0`.
- Building a vector database from scratch would take months. The `chromadb` library does it in one line.

### How we install libraries

Python libraries are stored on a website called **PyPI** (Python Package Index — `pypi.org`). The `pip` tool downloads them:

```bash
# Install a single library
pip install openai

# Install multiple libraries at once
pip install openai datasets chromadb sentence-transformers

# Install torch with CPU-only support (special URL, not on PyPI)
pip install torch --index-url https://download.pytorch.org/whl/cpu
```

### The libraries we installed and what each one does

In [10]:
# Let's import each library and show what it does

print("=" * 55)
print("LIBRARY INVENTORY")
print("=" * 55)

libraries = {
    "openai":               "HTTP client for calling Groq (OpenAI-compatible API)",
    "datasets":             "Downloads benchmark datasets from HuggingFace",
    "chromadb":             "Vector database — stores text as searchable number arrays",
    "sentence_transformers":"Converts text to number arrays (embeddings)",
    "torch":                "Deep learning framework (CPU-only build for this machine)",
    "matplotlib":           "Creates charts and graphs",
    "streamlit":            "Turns Python scripts into interactive web dashboards",
    "plotly":               "Interactive charts for the dashboard",
    "loguru":               "Structured logging — records what the system is doing",
}

import importlib
all_ok = True

for lib_name, description in libraries.items():
    try:
        module = importlib.import_module(lib_name)
        version = getattr(module, "__version__", "(version unknown)")
        print(f"  ✓ {lib_name:<25} v{version}")
        print(f"      → {description}")
    except ImportError:
        print(f"  ✗ {lib_name:<25} NOT INSTALLED")
        print(f"      → {description}")
        all_ok = False

print()
if all_ok:
    print("✓ All libraries are installed and importable")
else:
    print("⚠  Some libraries are missing — check your venv is active")

LIBRARY INVENTORY
  ✓ openai                    v1.109.1
      → HTTP client for calling Groq (OpenAI-compatible API)


/run/media/bala/HDD/Projects/scigraphagent-benchmark/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


  ✓ datasets                  v5.0.1
      → Downloads benchmark datasets from HuggingFace
  ✓ chromadb                  v1.5.9
      → Vector database — stores text as searchable number arrays
  ✓ sentence_transformers     v6.0.0
      → Converts text to number arrays (embeddings)
  ✓ torch                     v2.13.0+cpu
      → Deep learning framework (CPU-only build for this machine)
  ✓ matplotlib                v3.11.1
      → Creates charts and graphs
  ✓ streamlit                 v1.62.0
      → Turns Python scripts into interactive web dashboards
  ✓ plotly                    v7.0.0
      → Interactive charts for the dashboard
  ✓ loguru                    v0.7.3
      → Structured logging — records what the system is doing

✓ All libraries are installed and importable


---
## Section 7 — What is an API?

### The analogy

Think of a restaurant. You (the customer) do not go into the kitchen and cook your own food. Instead, you give your order to a waiter. The waiter takes it to the kitchen, the kitchen prepares the food, and the waiter brings it back to you.

An **API (Application Programming Interface)** works the same way:
- **You** = your Python code
- **The waiter** = the API
- **The kitchen** = a powerful computer running an AI model
- **The food** = the AI's response

You send a **request** (your order). The API returns a **response** (the food).

### How an API request works

1. Your code sends a request over the internet to a URL (like `https://api.groq.com/openai/v1/chat/completions`)
2. You include your **API key** — a secret password that proves who you are
3. You include your **message** — what you want the AI to do
4. The server receives the request, runs the AI model, and sends back the response
5. Your code reads the response and uses it

### What is an API key?

An API key is a unique secret string (like `gsk_GWHqh...`) that:
- Identifies who is making the request (your account)
- Is used to count how many requests you make (rate limiting)
- **Must never be shared or committed to GitHub** — anyone with your key can make requests charged to your account

### Environment variables — how we store the API key safely

Instead of writing the key directly in code (dangerous), we store it as an **environment variable** — a named value that the operating system holds in memory. The terminal command was:

```bash
export GROQ_API_KEY=gsk_your_key_here
```

The word `export` makes it available to any program started from this terminal, including Python.

In [11]:
import os

# os.environ is a dictionary of all environment variables on your system
# os.environ.get() looks up a variable by name and returns None if not found
api_key = os.environ.get("GROQ_API_KEY")

if api_key:
    # Show only the first 8 characters — enough to confirm it is set
    # without exposing the full key
    masked = api_key[:8] + "*" * (len(api_key) - 8)
    print("✓ GROQ_API_KEY is set")
    print(f"  Value (masked): {masked}")
    print(f"  Length: {len(api_key)} characters")
else:
    print("⚠  GROQ_API_KEY is NOT set")
    print("   In your terminal, run:")
    print("   export GROQ_API_KEY=gsk_your_key_here")
    print("   Then restart the Jupyter kernel and run this cell again")

⚠  GROQ_API_KEY is NOT set
   In your terminal, run:
   export GROQ_API_KEY=gsk_your_key_here
   Then restart the Jupyter kernel and run this cell again


---
## Section 8 — What is an LLM?

### What is machine learning?

Traditional software follows rules you write explicitly: "if the email contains the word 'winner', mark it as spam." Machine learning takes a different approach: show the software thousands of examples of spam and non-spam emails, and let it figure out the rules itself.

### What is a neural network?

A neural network is loosely inspired by the human brain. It consists of layers of simple mathematical units (called neurons) that each take some numbers in, multiply them by weights, and pass the result forward. Training a neural network means adjusting all those weights until the network produces correct outputs.

### What is a Large Language Model (LLM)?

An LLM is a very large neural network trained on vast amounts of text — books, websites, papers, code. By predicting what the next word should be billions of times across billions of sentences, it learns:
- Grammar and writing style
- Facts about the world
- How to reason and solve problems
- How to follow instructions

"Large" refers to the number of parameters (weights) in the network. GPT-OSS 120B has **120 billion parameters**. Storing and computing with 120 billion numbers requires enormous hardware — which is why we use Groq's servers instead of running it on your laptop.

### What is a prompt?

A prompt is the text you send to an LLM. The LLM reads the prompt and generates a response. In this project, we send two kinds of prompts:

1. **Generation prompts** — "Here is context from papers. Answer this question: ..."
2. **Judge prompts** — "Here is a question, context, and answer. Rate how faithful the answer is to the context on a scale of 0 to 1."

### Tokens — the unit of LLM processing

LLMs do not read words — they read **tokens**. A token is roughly 3-4 characters of English text. The word "knowledge" is one token. The word "unbelievable" is three tokens: "un", "believ", "able".

Tokens matter because:
- API pricing is per million tokens
- Rate limits are in tokens per minute
- Context window limits are in tokens (how much text an LLM can read at once)

In [12]:
# Let's demonstrate token estimation

def estimate_tokens(text):
    """Rough token estimate: 1 token ≈ 4 characters in English."""
    return max(1, len(text) // 4)

def token_cost(tokens, model="gpt-oss-120b"):
    """Estimate cost in USD for a given number of tokens."""
    # Groq pricing (August 2026):
    # GPT-OSS 120B: $0.15 per million input tokens, $0.60 per million output tokens
    prices = {
        "gpt-oss-120b": {"input": 0.15, "output": 0.60},
        "gpt-oss-20b":  {"input": 0.075, "output": 0.30},
    }
    price = prices.get(model, {"input": 0.15, "output": 0.60})
    # Assume 80% input, 20% output split
    input_tokens  = int(tokens * 0.8)
    output_tokens = int(tokens * 0.2)
    cost = (input_tokens * price["input"] + output_tokens * price["output"]) / 1_000_000
    return cost

# Sample texts of increasing length
texts = [
    "What is a knowledge graph?",
    "What is a knowledge graph and how does it help with multi-hop reasoning in RAG systems?",
    "A knowledge graph is a structured representation of information as a network of entities and relationships. " * 5,
]

print(f"{'Text preview':<50} {'Tokens':>8} {'Cost ($)':>12}")
print("-" * 74)
for text in texts:
    tokens = estimate_tokens(text)
    cost   = token_cost(tokens)
    preview = text[:47] + "..." if len(text) > 50 else text
    print(f"{preview:<50} {tokens:>8,} {cost:>12.8f}")

print()
print("For our n=50 benchmark run:")
total_tokens = 390_000
print(f"  Estimated total tokens: {total_tokens:,}")
print(f"  Cost on free tier: $0.00 (within Groq free limits)")
print(f"  Daily token budget used: {total_tokens/200_000*100:.0f}% of 200,000 TPD")

Text preview                                         Tokens     Cost ($)
--------------------------------------------------------------------------
What is a knowledge graph?                                6   0.00000120
What is a knowledge graph and how does it help ...       21   0.00000480
A knowledge graph is a structured representatio...      135   0.00003240

For our n=50 benchmark run:
  Estimated total tokens: 390,000
  Cost on free tier: $0.00 (within Groq free limits)
  Daily token budget used: 195% of 200,000 TPD


---
## Section 9 — What is Groq and why are we using it?

### The hardware problem

Running GPT-OSS 120B locally requires 80 GB of GPU memory. Your laptop may not have GPU or sufficient memory. Running this model locally may be physically impossible on such a machine.

### What is Groq?

Groq is a company that built custom chips called **LPUs (Language Processing Units)** specifically designed to run LLMs extremely fast. Their inference speed is 5-10x faster than typical GPU-based providers.

Groq offers a **free tier** with no credit card required:
- `openai/gpt-oss-120b` — our answer generator (matches o4-mini quality)
- `openai/gpt-oss-20b`  — our faithfulness judge (1,000 tokens/second)
- Limits: 30 requests/minute, 1,000 requests/day, 8,000 tokens/minute, 200,000 tokens/day

### Why not use Claude directly?

Claude API requires a separate paid account (not included in the Claude Pro subscription). For development runs (n=50), Groq is free. When we run the final paper benchmark (n=1,000), we will use Claude API for higher quality judging.

### The OpenAI-compatible interface

Groq's API speaks the same language as OpenAI's API. This means the `openai` Python library works with Groq by simply changing the `base_url`:

```python
# OpenAI:          base_url="https://api.openai.com/v1"
# Groq:            base_url="https://api.groq.com/openai/v1"
# Only ONE line changes. Everything else is identical.
```

In [ ]:
# ── Load API key from .env file ───────────────────────────────────
# The .env file stores secret keys that should never appear in code.
# load_dotenv() reads that file and puts the variables into os.environ
# so that os.environ.get("GROQ_API_KEY") works in all cells below.
#
# This must run before any cell that creates an OpenAI/Groq client.

from dotenv import load_dotenv
import os

load_dotenv()  # reads .env in the current directory
# huggingface_hub reads HF_TOKEN automatically from os.environ


api_key = os.environ.get("GROQ_API_KEY")
if api_key:
    print("✓ GROQ_API_KEY loaded successfully")
    print(f"  Starts with: {api_key[:8]}****")
else:
    print("✗ GROQ_API_KEY not found")
    print("  Make sure .env exists with: GROQ_API_KEY=gsk_...")

In [ ]:
import os
from openai import OpenAI

# ── Create the Groq client ────────────────────────────────────────
# OpenAI() creates a client object.
# We override the base_url to point at Groq instead of OpenAI.
# The api_key comes from the environment variable we set earlier.

client = OpenAI(
    base_url="https://api.groq.com/openai/v1",
    api_key=os.environ.get("GROQ_API_KEY"),
)

# ── Make a test call ──────────────────────────────────────────────
# client.chat.completions.create() sends a request to the LLM.
#
# The 'messages' list is the conversation:
#   - 'system' message: instructions for the AI (its personality and rules)
#   - 'user' message: what we are asking
#
# max_tokens limits how long the response can be.

print("Sending test request to Groq...")

response = client.chat.completions.create(
    model="openai/gpt-oss-120b",
    max_tokens=100,
    messages=[
        {
            "role":    "system",
            "content": "You are a helpful scientific assistant. Be concise."
        },
        {
            "role":    "user",
            "content": "In one sentence: what is retrieval-augmented generation?"
        }
    ]
)

# ── Read the response ─────────────────────────────────────────────
# response.choices is a list of possible responses (we always use the first one)
# .message.content is the actual text the model generated
answer = response.choices[0].message.content.strip()

print()
print("Model used:      ", response.model)
print("Input tokens:    ", response.usage.prompt_tokens)
print("Output tokens:   ", response.usage.completion_tokens)
print("Total tokens:    ", response.usage.total_tokens)
print()
print("Response:")
print(" ", answer)

OpenAIError: The api_key client option must be set either by passing api_key to the client or by setting the OPENAI_API_KEY environment variable

In [ ]:
# Now test the judge model (gpt-oss-20b)
# This is the model that evaluates answer quality in our benchmark.

context = """
Retrieval-Augmented Generation (RAG) was introduced by Lewis et al. in 2020.
It combines a retrieval system with a language model.
The retrieval system finds relevant documents.
The language model generates answers grounded in those documents.
"""

question = "Who introduced RAG and when?"
answer   = "RAG was introduced by Lewis et al. in 2020."

# This is exactly the type of prompt our EvalHarness sends to the judge
judge_prompt = f"""You are evaluating whether an answer is faithful to the context.

Context:
{context.strip()}

Question: {question}
Answer: {answer}

Is every claim in the answer supported by the context?
Return ONLY a JSON object: {{"faithfulness": <float 0.0-1.0>, "reason": "<one sentence>"}}"""

print("Sending faithfulness evaluation to judge model...")
judge_response = client.chat.completions.create(
    model="openai/gpt-oss-20b",   # smaller, faster model for judging
    max_tokens=100,
    messages=[{"role": "user", "content": judge_prompt}]
)

import json
raw = judge_response.choices[0].message.content.strip()
print("\nJudge raw response:", raw)

# Parse the JSON response
try:
    # Remove markdown code fences if present (```json ... ```)
    clean = raw.replace("```json", "").replace("```", "").strip()
    result = json.loads(clean)
    print(f"\nFaithfulness score: {result['faithfulness']}")
    print(f"Reason: {result['reason']}")
    if result['faithfulness'] >= 0.75:
        print("\n✓ Score meets threshold (≥ 0.75) — answer would be accepted")
    else:
        print("\n↺ Score below threshold — retry gate would trigger")
except json.JSONDecodeError:
    print("Could not parse JSON — raw response shown above")

---
## Section 10 — Verifying your complete setup

This section runs a comprehensive check of everything we have set up. Every item should show ✓.

In [ ]:
import sys, os, subprocess, importlib
from pathlib import Path

checks = []

def check(name, condition, detail=""):
    """Record a verification check."""
    checks.append((name, condition, detail))
    status = "✓" if condition else "✗"
    print(f"  {status}  {name}")
    if detail:
        print(f"       {detail}")

print("=" * 55)
print("  COMPLETE SETUP VERIFICATION")
print("=" * 55)

# ── Python ────────────────────────────────────────────────────────
print("\n[Python]")
py_version = sys.version_info
check("Python 3.10 or higher",
      py_version >= (3, 10),
      f"Found: {sys.version.split()[0]}")

check("Running in virtual environment",
      (Path(sys.prefix) / "pyvenv.cfg").exists(),
      f"Location: {sys.prefix}")

# ── Libraries ─────────────────────────────────────────────────────
print("\n[Libraries]")
required_libs = [
    "openai", "datasets", "chromadb",
    "sentence_transformers", "torch",
    "matplotlib", "streamlit", "plotly", "loguru"
]
for lib in required_libs:
    try:
        m = importlib.import_module(lib)
        v = getattr(m, "__version__", "?")
        check(f"{lib}", True, f"v{v}")
    except ImportError:
        check(f"{lib}", False, "Not installed")

# OpenAI version check (must be 1.x, not 3.x)
import openai
major = int(openai.__version__.split(".")[0])
check("openai version is 1.x (not 3.x)",
      major == 1,
      f"Found: {openai.__version__}")

# Torch CPU-only check
import torch
check("torch is CPU-only build",
      "+cpu" in torch.__version__,
      f"Version: {torch.__version__}")

# ── Environment variables ─────────────────────────────────────────
print("\n[Environment]")
api_key = os.environ.get("GROQ_API_KEY", "")
check("GROQ_API_KEY is set",
      bool(api_key),
      f"Starts with: {api_key[:8]}..." if api_key else "Not set")
check("API key format is valid (starts with gsk_)",
      api_key.startswith("gsk_"),
      "")

# ── Git ───────────────────────────────────────────────────────────
print("\n[Git]")
git_version = subprocess.run(
    "git --version", shell=True,
    capture_output=True, text=True
).stdout.strip()
check("Git is installed", bool(git_version), git_version)

git_remote = subprocess.run(
    "git remote get-url origin", shell=True,
    capture_output=True, text=True
).stdout.strip()
check("GitHub remote is connected",
      "github.com" in git_remote,
      git_remote)

# ── Folder structure ──────────────────────────────────────────────
print("\n[Project structure]")
root = Path(".")
check(".gitignore exists",  (root / ".gitignore").exists())
check("README.md exists",   (root / "README.md").exists())
check(".env.example exists",(root / ".env.example").exists())
check("requirements.txt exists", (root / "requirements.txt").exists())
check("data/ folder exists",   (root / "data").is_dir())
check("index/ folder exists",  (root / "index").is_dir())
check("results/ folder exists",(root / "results").is_dir())

# ── Groq API live check ───────────────────────────────────────────
print("\n[Groq API live check]")
try:
    from openai import OpenAI
    c = OpenAI(
        base_url="https://api.groq.com/openai/v1",
        api_key=api_key
    )
    models = c.models.list()
    model_ids = [m.id for m in models.data]
    check("Groq API responds", True, f"{len(model_ids)} models available")
    check("openai/gpt-oss-120b is live", "openai/gpt-oss-120b" in model_ids)
    check("openai/gpt-oss-20b is live",  "openai/gpt-oss-20b"  in model_ids)
except Exception as e:
    check("Groq API responds", False, str(e)[:60])

# ── Final summary ─────────────────────────────────────────────────
passed = sum(1 for _, ok, _ in checks if ok)
total  = len(checks)
print()
print("=" * 55)
print(f"  RESULT: {passed}/{total} checks passed")
if passed == total:
    print("  ✓ Setup is complete. Ready for Step 01.")
else:
    print("  ⚠  Fix the failing checks before proceeding.")
print("=" * 55)

---
## Section 11 — Understanding the project folder structure

Every file and folder in this project has a specific purpose. Understanding the structure helps you know where to look when something goes wrong.

```
scigraphagent-benchmark/
│
├── .venv/                  ← Virtual environment (hidden, never committed)
│   └── lib/python3.14/     ← All installed libraries live here
│
├── .git/                   ← Git internals (managed automatically, never edit)
│
├── data/                   ← Benchmark datasets (downloaded from HuggingFace)
│   ├── hotpotqa_sample_50.json
│   ├── musique_sample_50.json
│   └── manifest.json
│
├── index/                  ← ChromaDB vector store (built from data/)
│   └── chroma/
│
├── results/                ← Experiment outputs
│   ├── raw_results_hotpotqa_50.json
│   ├── metrics_hotpotqa_50.json
│   └── figures/            ← PNG charts
│
├── .gitignore              ← Tells Git what NOT to track
├── .env.example            ← Shows which environment variables are needed
├── README.md               ← Project description
├── requirements.txt        ← Pinned library versions
│
├── step01_load_benchmarks.py       ← Downloads datasets
├── step02_build_retrieval_systems.py← Builds vector index + knowledge graph
├── step03_run_experiments.py       ← Runs 3 benchmark experiments via Groq
├── step04_compute_metrics.py       ← Aggregates and prints results
├── step05_visualise.py             ← Generates charts
├── step06_dashboard.py             ← Streamlit interactive dashboard
└── run_all.py                      ← Runs all steps in sequence
```

### Why some folders are greyed out in VS Code

`.venv/`, `data/`, `index/`, and `results/` are listed in `.gitignore`. VS Code can be configured to hide gitignored files from the Explorer panel (which we did in Settings). They still exist on disk — they are just hidden from the view to reduce clutter.

In [ ]:
from pathlib import Path

def show_tree(root, indent=0, skip=None):
    """Print a simple folder tree."""
    skip = skip or {".git", ".venv", "__pycache__", ".ipynb_checkpoints"}
    items = sorted(Path(root).iterdir(), key=lambda p: (p.is_file(), p.name))
    for item in items:
        if item.name in skip:
            continue
        prefix = "    " * indent + ("├── " if indent else "")
        if item.is_dir():
            print(f"{prefix}{item.name}/")
            show_tree(item, indent + 1, skip)
        else:
            size = item.stat().st_size
            if size > 1024:
                size_str = f"({size // 1024} KB)"
            else:
                size_str = f"({size} B)"
            print(f"{prefix}{item.name:<40} {size_str}")

print("Project structure:")
show_tree(".")

### Understanding `.gitignore`

Let's read and explain our `.gitignore` file line by line:

In [ ]:
from pathlib import Path

gitignore_path = Path(".gitignore")

# Explanations for each rule in our .gitignore
explanations = {
    ".venv/":        "Virtual environment — 500MB of packages, re-creatable from requirements.txt",
    ".env":          "Contains real API keys — NEVER commit secrets to GitHub",
    "*.env":         "Any file ending in .env — same reason",
    "data/":         "Large datasets — re-downloadable from HuggingFace",
    "index/":        "ChromaDB vector store — rebuilds in seconds from data/",
    "results/":      "Experiment outputs — commit selectively, not automatically",
    "__pycache__/":  "Python cache files — auto-generated, not needed in repo",
    ".vscode/":      "VS Code workspace settings — personal preferences",
}

print(".gitignore contents and explanations:\n")
with open(gitignore_path) as f:
    for line in f:
        line = line.strip()
        if not line or line.startswith("#"):
            if line.startswith("#"):
                print(f"  {line}")  # Show comments
            continue
        explanation = explanations.get(line, "")
        print(f"  {line:<25}", end="")
        if explanation:
            print(f" ← {explanation}")
        else:
            print()

---
## Section 12 — What comes next?

### What we have built so far

In this notebook we set up the complete foundation:

| Component | Status | What it does |
|---|---|---|
| Python 3.14.4 | ✓ Ready | The programming language |
| Virtual environment | ✓ Active | Isolated Python for this project |
| All libraries | ✓ Installed | Tools for AI, data, and visualisation |
| Git | ✓ Configured | Version control — tracks all changes |
| GitHub repo | ✓ Connected | Online backup at github.com/DrBala-2-0 |
| Groq API key | ✓ Set | Free access to GPT-OSS 120B and 20B |
| API calls tested | ✓ Working | Both generator and judge models respond |
| Project structure | ✓ Created | Organised folders for data, code, results |

### What comes in the next notebooks

**Notebook 01 — Loading Benchmark Datasets**  
We download HotpotQA, MuSiQue, and 2WikiMultiHopQA from HuggingFace and explore their structure. You will learn about multi-hop questions and why they are hard for standard AI systems.

**Notebook 02 — Building Retrieval Systems**  
We build a vector index (ChromaDB) and a knowledge graph from the dataset passages. You will learn what embeddings are, what a knowledge graph is, and how BFS traversal works.

**Notebook 03 — Running Experiments**  
We run the three benchmark experiments and see the RAGAS retry gate in action. You will learn about faithfulness scoring, the retry mechanism, and how to measure quality.

**Notebook 04 — Metrics and Results**  
We compute F1, Exact Match, faithfulness, and recall lift. You will learn how to interpret benchmark results and what the numbers mean.

**Notebook 05 — Visualisation**  
We generate charts comparing all four retrieval conditions. You will learn how to read ablation study results.

### The key insight to carry forward

Everything in this project follows the same pattern:

```
1. Read data from disk (or download it)
2. Process it (extract entities, build graphs, create embeddings)
3. Call the Groq API with a prompt
4. Parse the response
5. Save results to disk
6. Commit to Git
```

Each step does exactly one thing and saves its output for the next step. This modular design means if step 3 fails, you do not lose the work from steps 1 and 2.

In [ ]:
import subprocess

# Final check — show the current git state
print("Current Git state:")
print()

log = subprocess.run(
    "git log --oneline",
    shell=True, capture_output=True, text=True
).stdout.strip()

print("Commits on GitHub:")
for line in log.split("\n"):
    print("  ", line)

print()
status = subprocess.run(
    "git status --short",
    shell=True, capture_output=True, text=True
).stdout.strip()

if status:
    print("Uncommitted changes:")
    for line in status.split("\n"):
        print("  ", line)
    print()
    print("Tip: commit this notebook with:")
    print("  git add 00_setup_and_foundations.ipynb")
    print('  git commit -m "Add setup notebook: foundations and verification"')
    print("  git push")
else:
    print("✓ All changes committed and synced with GitHub")

---
## Committing this notebook to GitHub

After running all cells successfully, save this notebook (`Ctrl+S`) and commit it:

```bash
git add 00_setup_and_foundations.ipynb
git commit -m "Add Notebook 00: setup and foundations walkthrough"
git push
```

**Best practice:** Commit notebooks after each session — not after every single cell. A good commit captures a meaningful unit of work, like completing a notebook or a working step.

---

*Notebook 00 complete. Continue with `01_load_benchmarks.ipynb` after running `step01_load_benchmarks.py --n 3` successfully.*